# 01 — Splits and augmentation
Splits the raw 10K **first**, then augments **only the training images**. Test and validation images never enter augmentation, so nothing leaks.

Run all cells in order. Needs: `pip install pandas opencv-python albumentations tqdm`.

In [ ]:
import json, os, re, subprocess, sys, shutil
import pandas as pd
import config as C
for d in (C.SPLIT_DIR, C.AUG_DIR, C.SUBSET_DIR, C.MODELS_DIR, C.RESULTS_DIR, C.FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)      # recreate folders if you deleted them

def norm(t): return re.sub(r"\s+", " ", str(t)).strip()

df = pd.read_csv(C.RAW_CSV)
print("Columns in your CSV:", list(df.columns))

# The image-name column may be called file_name, image, filename, ... -> normalise to "file_name"
for cand in ("file_name", "filename", "image", "image_name", "img", "path", "image_path"):
    if cand in df.columns:
        if cand != "file_name":
            df = df.rename(columns={cand: "file_name"})
            print(f"Renamed column '{cand}' -> 'file_name'")
        break
else:
    raise KeyError("No image-name column found. Look at the printed column list above and add its name to the tuple.")
df["file_name"] = df["file_name"].astype(str).map(os.path.basename)   # keep only the file name

need = ["id", "text", "bbox", "lines", "tokens", "theme"]
absent = [c for c in need if c not in df.columns]
assert not absent, f"CSV is missing columns: {absent}"

print(f"Raw rows: {len(df):,}")
assert df["id"].is_unique
missing = [f for f in df["file_name"] if not (C.RAW_IMG_DIR / f).exists()]
print("Missing image files:", len(missing))
assert not missing, missing[:5]

## Split into train / val / test (with duplicate-text removal)

In [ ]:
test  = df.sample(n=C.N_TEST, random_state=C.SEED)
rest  = df.drop(test.index)
val   = rest.sample(n=C.N_VAL, random_state=C.SEED)
train = rest.drop(val.index)

# a synthetic benchmark can reuse sentences; make sure no text is shared across splits
test_txt = set(test["text"].map(norm))
n_tr, n_va = len(train), len(val)
val   = val[~val["text"].map(norm).isin(test_txt)]
train = train[~train["text"].map(norm).isin(test_txt)]
train = train[~train["text"].map(norm).isin(set(val["text"].map(norm)))]
print(f"Removed for shared text -> train: {n_tr-len(train)}, val: {n_va-len(val)}")

ids = [set(x["id"]) for x in (train, val, test)]
assert not (ids[0] & ids[1]) and not (ids[0] & ids[2]) and not (ids[1] & ids[2])

for name, part in (("train", train), ("val", val), ("test", test)):
    part.sort_values("id").to_csv(C.SPLIT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
    print(f"{name:5s}: {len(part):>6,}")
json.dump({k: sorted(map(int, v["id"])) for k, v in (("train", train), ("val", val), ("test", test))},
          open(C.SPLIT_DIR / "splits.json", "w"))

## Augmentation library (written to `aug_lib.py`)
Stage 1 replaces the background by alpha-matting; stage 2 applies bbox-aware photometric and geometric changes.

In [ ]:
%%writefile aug_lib.py
import random, json, ast
import numpy as np
import cv2
import albumentations as A

A_MAJOR = int(A.__version__.split(".")[0])

def _light():
    if random.random() < 0.7:
        v = random.randint(210, 250); return (v, v, v)
    return tuple(random.randint(210, 250) for _ in range(3))

def _dark():
    if random.random() < 0.7:
        v = random.randint(10, 60); return (v, v, v)
    return tuple(random.randint(10, 60) for _ in range(3))

def make_solid(w, h, theme):
    bg = np.zeros((h, w, 3), np.uint8); bg[:] = _light() if theme == "light" else _dark(); return bg

def make_gradient(w, h, theme):
    base, amp = (235, 20) if theme == "light" else (30, 20)
    axis = random.choice(["x", "y", "diag"])
    yy, xx = np.mgrid[0:h, 0:w]
    if axis == "x": g = xx / max(w - 1, 1)
    elif axis == "y": g = yy / max(h - 1, 1)
    else: g = (xx / max(w - 1, 1) + yy / max(h - 1, 1)) / 2.0
    g = g * 2 - 1
    if random.random() < 0.5: g = -g
    v = np.clip(base + amp * g, 0, 255).astype(np.uint8)
    return np.stack([v, v, v], -1)

def make_paper(w, h, theme):
    coarse = np.random.rand(max(h // 8, 4), max(w // 8, 4)).astype(np.float32)
    coarse = cv2.resize(coarse, (w, h), interpolation=cv2.INTER_CUBIC)
    tex = coarse * 0.6 + np.random.rand(h, w).astype(np.float32) * 0.15
    tex = (tex - tex.min()) / (tex.max() - tex.min() + 1e-6)
    v = (225 + tex * 30).astype(np.uint8) if theme == "light" else (15 + tex * 45).astype(np.uint8)
    return np.stack([v, v, v], -1)

def make_grid(w, h, theme):
    bgv, lv = (245, 200) if theme == "light" else (30, 60)
    bg = np.full((h, w, 3), bgv, np.uint8)
    step = random.choice([20, 30, 40, 50])
    for y in range(0, h, step): cv2.line(bg, (0, y), (w, y), (lv,) * 3, 1)
    for x in range(0, w, step): cv2.line(bg, (x, 0), (x, h), (lv,) * 3, 1)
    return bg

_BG = [make_solid, make_gradient, make_paper, make_paper, make_grid]   # paper texture weighted 2x
def generate_background(w, h, theme): return random.choice(_BG)(w, h, theme)

def replace_background(img, new_bg, theme):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)
    if theme == "light":
        alpha = 1.0 - gray / 255.0
        sel = gray < np.percentile(gray, 5)
        tc = img[sel].mean(axis=0) if sel.any() else np.array([0, 0, 0], np.float32)
    else:
        alpha = gray / 255.0
        sel = gray > np.percentile(gray, 95)
        tc = img[sel].mean(axis=0) if sel.any() else np.array([255, 255, 255], np.float32)
    lo, hi = float(alpha.min()), float(alpha.max())
    if hi - lo > 1e-6: alpha = (alpha - lo) / (hi - lo)
    alpha = alpha[..., None]
    layer = np.broadcast_to(tc, img.shape).astype(np.float32)
    return np.clip(alpha * layer + (1 - alpha) * new_bg.astype(np.float32), 0, 255).astype(np.uint8)

# ---------- augmentation transforms (version-aware; nothing here can hide or blank the text) ----------
def _soft_light(image, **kwargs):
    # smooth, low-frequency brightness variation (+-12%) instead of hard dark polygon shadows
    h, w = image.shape[:2]
    field = cv2.resize(np.random.rand(3, 4).astype(np.float32), (w, h), interpolation=cv2.INTER_CUBIC)
    field = 1.0 + (field - 0.5) * 0.24
    return np.clip(image.astype(np.float32) * field[..., None], 0, 255).astype(np.uint8)

def _noise(p):
    if A_MAJOR >= 2: return A.GaussNoise(std_range=(0.02, 0.06), mean_range=(0.0, 0.0), p=p)   # about 5-15 grey levels
    return A.GaussNoise(var_limit=(10.0, 50.0), p=p)

def _affine():
    kw = dict(translate_percent={"x": (-0.02, 0.02), "y": (-0.02, 0.02)}, scale=(0.98, 1.02), rotate=(-1, 1), p=0.40)
    # BORDER_REPLICATE extends the background at the edges, so there are never black bars
    if A_MAJOR >= 2: return A.Affine(border_mode=cv2.BORDER_REPLICATE, **kw)
    return A.Affine(mode=cv2.BORDER_REPLICATE, **kw)

def _bbox_params(min_vis):
    return A.BboxParams(format="coco", label_fields=["box_labels"], min_visibility=min_vis, min_area=1.0)

def _compose(tfms, bp, seed):
    if A_MAJOR >= 2: return A.Compose(tfms, bbox_params=bp, seed=seed)
    return A.Compose(tfms, bbox_params=bp)

def build_pipeline(theme, seed=None):
    return _compose([
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.20, p=0.6),
        A.Lambda(image=_soft_light, p=0.30),
        A.RandomGamma(gamma_limit=(90, 110), p=0.30),
        A.GaussianBlur(blur_limit=(3, 5), p=0.25), _noise(0.25),
        A.ImageCompression(quality_range=(70, 95), p=0.30), _affine()], _bbox_params(0.4), seed)

def build_mild_pipeline(seed=None):
    return _compose([A.RandomBrightnessContrast(p=0.5), A.Lambda(image=_soft_light, p=0.3), A.GaussianBlur(p=0.2)],
                    _bbox_params(0.1), seed)

def parse_annotation(raw):
    if raw is None or (isinstance(raw, float) and np.isnan(raw)): return []
    if isinstance(raw, (list, dict)): return raw
    s = str(raw).strip()
    if not s or s.lower() == "nan": return []
    try: return json.loads(s)
    except Exception:
        try: return ast.literal_eval(s)
        except Exception: return []


## Worker (written to `aug_worker.py`)
Every variant is seeded from `(source id, variant index)`, so the output is identical however the work is sharded.

In [ ]:
%%writefile aug_worker.py
import sys, json, random
import numpy as np, pandas as pd, cv2
from tqdm import tqdm
import config as C
from aug_lib import generate_background, replace_background, parse_annotation
from aug_quality import augment_image

SHARD = int(sys.argv[1])
OUT_DIR = C.AUG_DIR / "shards" / f"shard{SHARD}"
OUT_IMG = OUT_DIR / "images"
OUT_IMG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(C.SPLIT_DIR / "train.csv")            # TRAIN ONLY
shard_df = df.iloc[SHARD::C.N_SHARDS]
print(f"[shard {SHARD}] {len(shard_df)} sources x {C.MULTIPLIER}", flush=True)

rows = []
for _, row in tqdm(shard_df.iterrows(), total=len(shard_df), desc=f"shard{SHARD}"):
    src_id = int(row["id"])
    img = cv2.imread(str(C.RAW_IMG_DIR / row["file_name"]), cv2.IMREAD_COLOR)
    if img is None: continue
    h, w = img.shape[:2]
    theme = str(row.get("theme", "light")).lower()
    theme = theme if theme in ("light", "dark") else "light"

    main = parse_annotation(row.get("bbox"))
    if not (isinstance(main, list) and len(main) == 4): continue
    boxes, meta = [[float(v) for v in main]], [("main", None)]
    for key, kind in (("lines", "line"), ("tokens", "token")):
        items = parse_annotation(row.get(key))
        if isinstance(items, list):
            for it in items:
                if isinstance(it, dict) and isinstance(it.get("bbox"), list) and len(it["bbox"]) == 4:
                    boxes.append([float(v) for v in it["bbox"]]); meta.append((kind, it))
    labels = list(range(len(boxes)))

    for k in range(C.MULTIPLIER):
        try:
            s = C.SEED * 1_000_003 + src_id * 1000 + k
            random.seed(s); np.random.seed(s)
            comp = replace_background(img, generate_background(w, h, theme), theme)
            out_img, out_boxes, out_labels, applied = augment_image(img, comp, theme, boxes, labels, s)

            new_main, new_lines, new_toks = None, [], []
            for b, lab in zip(out_boxes, out_labels):
                kind, m = meta[int(lab)]
                b = [float(v) for v in b]
                if kind == "main": new_main = b
                else:
                    m = dict(m); m["bbox"] = b
                    (new_lines if kind == "line" else new_toks).append(m)
            if new_main is None: continue

            fn = f"s{SHARD}_aug_{src_id:06d}_{k:03d}.jpg"
            cv2.imwrite(str(OUT_IMG / fn), out_img, [cv2.IMWRITE_JPEG_QUALITY, 95])
            r = row.to_dict()
            r.update(source_id=src_id, aug_index=k, file_name=fn, bbox=json.dumps(new_main),
                     lines=json.dumps(new_lines, ensure_ascii=False), tokens=json.dumps(new_toks, ensure_ascii=False),
                     bg_color="", font_color="", theme=theme, photometric_applied=int(applied))
            rows.append(r)
        except Exception:
            continue

pd.DataFrame(rows).to_csv(OUT_DIR / "shard.csv", index=False, encoding="utf-8-sig")
print(f"[shard {SHARD}] DONE: {len(rows)} rows", flush=True)


## Quick sanity test before the full run
Look at the pictures: text must stay clearly readable and the boxes must sit on the text. The second cell measures how strong the noise is.

In [ ]:
import random, numpy as np, cv2, matplotlib.pyplot as plt
import importlib, aug_lib; importlib.reload(aug_lib)
from aug_lib import *
tr = pd.read_csv(C.SPLIT_DIR / "train.csv").head(3)
fig, ax = plt.subplots(3, 2, figsize=(12, 7))
for i, (_, r) in enumerate(tr.iterrows()):
    img = cv2.imread(str(C.RAW_IMG_DIR / r["file_name"])); h, w = img.shape[:2]
    theme = r["theme"] if r["theme"] in ("light", "dark") else "light"
    box = [float(v) for v in parse_annotation(r["bbox"])]
    random.seed(i); np.random.seed(i)
    comp = replace_background(img, generate_background(w, h, theme), theme)
    out = build_pipeline(theme, i)(image=comp, bboxes=[box], box_labels=[0])
    ax[i, 0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax[i, 0].axis("off")
    aug = cv2.cvtColor(out["image"], cv2.COLOR_BGR2RGB).copy()
    for b in out["bboxes"]:
        x, y, bw, bh = map(int, b); cv2.rectangle(aug, (x, y), (x + bw, y + bh), (255, 0, 0), 2)
    ax[i, 1].imshow(aug); ax[i, 1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
import albumentations as A
print("albumentations version:", A.__version__)
r0 = pd.read_csv(C.SPLIT_DIR / "train.csv").iloc[0]
im0 = cv2.imread(str(C.RAW_IMG_DIR / r0["file_name"]))
noisy = A.Compose([aug_lib._noise(1.0)])(image=im0)["image"]
print("mean absolute pixel change from noise:", round(float(np.abs(noisy.astype(float) - im0.astype(float)).mean()), 2), "(should be below about 12)")
fig, ax = plt.subplots(1, 2, figsize=(12, 3))
ax[0].imshow(cv2.cvtColor(im0, cv2.COLOR_BGR2RGB)); ax[0].set_title("original"); ax[0].axis("off")
ax[1].imshow(cv2.cvtColor(noisy, cv2.COLOR_BGR2RGB)); ax[1].set_title("with noise only"); ax[1].axis("off")
plt.show()

## Quality-gated augmentation (`aug_quality.py`)
Each variant is only kept if the text still has at least 70% of its original contrast. Otherwise it falls back to a clean background swap, so unreadable images cannot be produced.

In [ ]:
%%writefile aug_quality.py
import numpy as np, cv2
import albumentations as A
from aug_lib import A_MAJOR, _noise, _affine, _bbox_params, _compose

MIN_RATIO = 0.70      # augmented text must keep at least 70% of its contrast against the background
MAX_TRIES = 6

def _gentle_light(image, **kwargs):
    # smooth brightness variation, +-8%
    h, w = image.shape[:2]
    field = cv2.resize(np.random.rand(3, 4).astype(np.float32), (w, h), interpolation=cv2.INTER_CUBIC)
    field = 1.0 + (field - 0.5) * 0.16
    return np.clip(image.astype(np.float32) * field[..., None], 0, 255).astype(np.uint8)

def _plain_compose(tfms, seed):
    if A_MAJOR >= 2: return A.Compose(tfms, seed=seed)
    return A.Compose(tfms)

def _photometric():
    return [
        A.RandomBrightnessContrast(brightness_limit=(-0.10, 0.10), contrast_limit=(-0.10, 0.10), p=0.5),
        A.Lambda(image=_gentle_light, p=0.3),
        A.RandomGamma(gamma_limit=(90, 110), p=0.3),
        A.GaussianBlur(blur_limit=(3, 3), p=0.2),
        _noise(0.25),
        A.ImageCompression(quality_range=(75, 95), p=0.3),
    ]

def text_masks(img, theme):
    # text pixels / background pixels, found from the original image
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)
    alpha = (1.0 - gray / 255.0) if theme == "light" else (gray / 255.0)
    lo, hi = float(alpha.min()), float(alpha.max())
    alpha = (alpha - lo) / (hi - lo + 1e-6)
    return alpha > 0.8, alpha < 0.05

def contrast(img, tmask, bmask):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)
    return abs(float(g[bmask].mean()) - float(g[tmask].mean()))

def augment_image(orig, comp, theme, boxes, labels, seed):
    # returns (image, boxes, labels, photometric_applied)
    tmask, bmask = text_masks(orig, theme)
    img, applied = comp, False
    if tmask.any() and bmask.any():
        base = contrast(comp, tmask, bmask)
        for attempt in range(MAX_TRIES):
            cand = _plain_compose(_photometric(), seed + attempt)(image=comp)["image"]
            if contrast(cand, tmask, bmask) >= MIN_RATIO * base:     # text still clearly visible?
                img, applied = cand, True
                break
    # tiny geometric change with box tracking; if too many boxes are lost, skip the geometry
    out = _compose([_affine()], _bbox_params(0.4), seed)(image=img, bboxes=boxes, box_labels=labels)
    if len(out["bboxes"]) < max(1, int(0.85 * len(boxes))):
        return img, boxes, labels, applied
    return out["image"], out["bboxes"], out["box_labels"], applied


In [ ]:
import importlib, aug_lib, aug_quality
importlib.reload(aug_lib); importlib.reload(aug_quality)
from aug_lib import generate_background, replace_background, parse_annotation
from aug_quality import augment_image

samp = pd.read_csv(C.SPLIT_DIR / "train.csv").sample(12, random_state=1)     # 12 DIFFERENT source images
fig, ax = plt.subplots(4, 3, figsize=(14, 8))
n_applied = 0
for k, (a, (_, r)) in enumerate(zip(ax.ravel(), samp.iterrows())):
    im = cv2.imread(str(C.RAW_IMG_DIR / r["file_name"])); h, w = im.shape[:2]
    th = r["theme"] if r["theme"] in ("light", "dark") else "light"
    bx = [float(v) for v in parse_annotation(r["bbox"])]
    s = 100 + k; random.seed(s); np.random.seed(s)
    comp = replace_background(im, generate_background(w, h, th), th)
    o, _, _, applied = augment_image(im, comp, th, [bx], [0], s)
    n_applied += applied
    a.imshow(cv2.cvtColor(o, cv2.COLOR_BGR2RGB)); a.axis("off"); a.set_title("photometric applied" if applied else "background only", fontsize=8)
plt.tight_layout(); plt.show()
print(f"{n_applied}/12 got photometric changes; the rest fell back to a clean background swap (both are readable)")

## Run augmentation (8 shards, roughly 2 hours)
Workers run as separate processes. Progress is written to `shard_*.log` in `augmented/`.

In [ ]:
procs = []
for s in range(C.N_SHARDS):
    log = open(C.AUG_DIR / f"shard_{s}.log", "w", encoding="utf-8")
    procs.append((subprocess.Popen([sys.executable, "aug_worker.py", str(s)], stdout=log, stderr=subprocess.STDOUT), log))
    print("launched shard", s)
for p, log in procs:
    p.wait(); log.close()
print("All workers finished. Return codes:", [p.returncode for p, _ in procs])

## Merge shards and verify

In [ ]:
(C.AUG_FINAL / "images").mkdir(parents=True, exist_ok=True)
dfs = []
for s in range(C.N_SHARDS):
    sd = C.AUG_DIR / "shards" / f"shard{s}"
    if not (sd / "shard.csv").exists():
        print("WARNING: missing shard", s); continue
    dfs.append(pd.read_csv(sd / "shard.csv"))
    for f in (sd / "images").iterdir():
        shutil.move(str(f), str(C.AUG_FINAL / "images" / f.name))
merged = pd.concat(dfs, ignore_index=True)
merged["id"] = range(1, len(merged) + 1)
merged.to_csv(C.AUG_FINAL / "augmented.csv", index=False, encoding="utf-8-sig")

sp = json.load(open(C.SPLIT_DIR / "splits.json"))
src = set(merged["source_id"].astype(int))
assert not (src & set(sp["test"])), "LEAK: a test image was augmented"
assert not (src & set(sp["val"])),  "LEAK: a val image was augmented"
assert src <= set(sp["train"])
bad = [f for f in merged["file_name"].sample(min(2000, len(merged)), random_state=0) if not (C.AUG_FINAL / "images" / f).exists()]
assert not bad, bad[:3]
gb = sum(f.stat().st_size for f in (C.AUG_FINAL / "images").iterdir()) / 1e9
print(f"OK: {len(merged):,} augmented images | {len(src):,} unique source images | {gb:.1f} GB | no leakage")